In [1]:
import pandas as pd

# 1. Load the dataset
file_path = "EEG.machinelearing_data_BRMH.csv"
df = pd.read_csv(file_path)

# 2. Count the number of rows and columns
num_rows, num_cols = df.shape
print(f"Total number of rows (samples): {num_rows}")
print(f"Total number of columns (attributes): {num_cols}")
print("-" * 30)

FileNotFoundError: [Errno 2] No such file or directory: 'EEG.machinelearing_data_BRMH.csv'

In [3]:
# 3. Print all the column names
print("List of all columns in the dataset:")
column_names = list(df.columns)
print(column_names)
print("-" * 30)

# 4. Display dataset structure and data types
print("\nDataset Info:")
df.info()

List of all columns in the dataset:
['no.', 'sex', 'age', 'eeg.date', 'education', 'IQ', 'main.disorder', 'specific.disorder', 'AB.A.delta.a.FP1', 'AB.A.delta.b.FP2', 'AB.A.delta.c.F7', 'AB.A.delta.d.F3', 'AB.A.delta.e.Fz', 'AB.A.delta.f.F4', 'AB.A.delta.g.F8', 'AB.A.delta.h.T3', 'AB.A.delta.i.C3', 'AB.A.delta.j.Cz', 'AB.A.delta.k.C4', 'AB.A.delta.l.T4', 'AB.A.delta.m.T5', 'AB.A.delta.n.P3', 'AB.A.delta.o.Pz', 'AB.A.delta.p.P4', 'AB.A.delta.q.T6', 'AB.A.delta.r.O1', 'AB.A.delta.s.O2', 'AB.B.theta.a.FP1', 'AB.B.theta.b.FP2', 'AB.B.theta.c.F7', 'AB.B.theta.d.F3', 'AB.B.theta.e.Fz', 'AB.B.theta.f.F4', 'AB.B.theta.g.F8', 'AB.B.theta.h.T3', 'AB.B.theta.i.C3', 'AB.B.theta.j.Cz', 'AB.B.theta.k.C4', 'AB.B.theta.l.T4', 'AB.B.theta.m.T5', 'AB.B.theta.n.P3', 'AB.B.theta.o.Pz', 'AB.B.theta.p.P4', 'AB.B.theta.q.T6', 'AB.B.theta.r.O1', 'AB.B.theta.s.O2', 'AB.C.alpha.a.FP1', 'AB.C.alpha.b.FP2', 'AB.C.alpha.c.F7', 'AB.C.alpha.d.F3', 'AB.C.alpha.e.Fz', 'AB.C.alpha.f.F4', 'AB.C.alpha.g.F8', 'AB.C.alpha.

# PHASE 2 Data Preprocessing (Cleaning & Signal Processing)

In [2]:
# --- 1. Tabular Data Cleaning (Training Data) ---

# Drop any accidental 'Unnamed' columns
unnamed_cols = [col for col in df.columns if 'Unnamed' in col]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)
    print(f"Cleaned up {len(unnamed_cols)} empty/unnamed columns.\n")

# Check for missing values
total_rows = len(df)
rows_with_nulls = df.isnull().any(axis=1).sum()
percent_missing = (rows_with_nulls / total_rows) * 100

print(f"Total rows with at least one missing value: {rows_with_nulls} out of {total_rows} ({percent_missing:.2f}%)\n")

# Decision Logic: Drop vs. Impute
print("=== HANDLING MISSING DATA ===")
if percent_missing < 5.0:
    print("Decision: DROP")
    print("Since less than 5% of rows are affected, it is safe to drop them.")
    df_cleaned = df.dropna()
    print(f"Dataset size went from {total_rows} to {len(df_cleaned)} rows.")
else:
    print("Decision: IMPUTE")
    df_cleaned = df.copy()
    numeric_cols = df_cleaned.select_dtypes(include=['number']).columns
    df_cleaned[numeric_cols] = df_cleaned[numeric_cols].fillna(df_cleaned[numeric_cols].mean())
    df_cleaned = df_cleaned.dropna(subset=['main.disorder'])
    print("Missing values imputed with column means.")


# --- 2. Live Signal Preprocessing Placeholder (Campus Edge Server) ---
# In deployment, raw signals from BLE wearables will pass through this pipeline

def preprocess_live_eeg_stream(raw_eeg_signal):
    """
    Applies real-time preprocessing as defined in the Full-Stack Architecture.
    """
    # 1. signal = apply_bandpass_filter(raw_eeg_signal, lowcut=1.0, highcut=50.0)
    # 2. signal = apply_ica(signal)  # Independent Component Analysis
    # 3. clean_signal = remove_artifacts_faster(signal) # FASTER Algorithm
    # return clean_signal
    pass

NameError: name 'df' is not defined

In [5]:
from sklearn.preprocessing import LabelEncoder

# --- 3. Target Encoding & Feature Separation (Training Data) ---
print("=== FEATURE AND TARGET SEPARATION ===")

target_column = 'main.disorder'

# Initialize LabelEncoder to convert text labels to numbers (0 to 6)
le = LabelEncoder()
df_cleaned['target_encoded'] = le.fit_transform(df_cleaned[target_column])

# Print the mapping so we know which number corresponds to which disorder
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Disorder Label Mapping:")
for disorder, code in label_mapping.items():
    print(f"  {code}: {disorder}")

# Separate Features (X) and Target (y)
# We drop administrative/text columns to isolate the EEG and demographic features
cols_to_drop = ['no.', 'eeg.date', 'main.disorder', 'specific.disorder', 'target_encoded']
X = df_cleaned.drop(columns=cols_to_drop)
y = df_cleaned['target_encoded']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("-" * 40)

=== FEATURE AND TARGET SEPARATION ===
Disorder Label Mapping:
  0: Addictive disorder
  1: Anxiety disorder
  2: Healthy control
  3: Mood disorder
  4: Obsessive compulsive disorder
  5: Schizophrenia
  6: Trauma and stress related disorder

Features (X) shape: (919, 1144)
Target (y) shape: (919,)
----------------------------------------


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27312\1884415156.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['target_encoded'] = le.fit_transform(df_cleaned[target_column])


# Phase 3: Feature Extraction Layer

In [3]:
import numpy as np
import scipy.signal as signal

# ==========================================
# PHASE 3: LIVE SIGNAL FEATURE EXTRACTION 
# ==========================================

def extract_features_live_eeg(clean_eeg_left, clean_eeg_right=None, fs=256):
    """
    Transforms 1D time-series EEG data into 2D spectrograms and discrete biomarker vectors.
    """
    
    # 1. TRANSFORMATION: STFT to create Dynamic 2D Spectrograms
    freqs_stft, times_stft, Zxx = signal.stft(clean_eeg_left, fs=fs, nperseg=128)
    spectrogram_2d = np.abs(Zxx) 
    
    # 2. QUANTIFICATION: Power Spectral Density (PSD) via Welch's Method
    freqs_welch, psd_left = signal.welch(clean_eeg_left, fs=fs, nperseg=256)
    
    def get_band_power(freqs, psd, f_min, f_max):
        idx = np.logical_and(freqs >= f_min, freqs <= f_max)
        return np.sum(psd[idx])
        
    theta_power_L = get_band_power(freqs_welch, psd_left, 4, 8)
    alpha_power_L = get_band_power(freqs_welch, psd_left, 8, 12)
    beta_power_L = get_band_power(freqs_welch, psd_left, 13, 30)

    # 3. BIOMARKER GENERATION
    tbr_ratio = theta_power_L / beta_power_L if beta_power_L > 0 else 0
    
    alpha_asymmetry = None
    if clean_eeg_right is not None:
        _, psd_right = signal.welch(clean_eeg_right, fs=fs, nperseg=256)
        alpha_power_R = get_band_power(freqs_welch, psd_right, 8, 12)
        
        # Formula: (Right Alpha - Left Alpha) / (Right Alpha + Left Alpha)
        alpha_asymmetry = (alpha_power_R - alpha_power_L) / (alpha_power_R + alpha_power_L)

    # 4. OUTPUT: Structured Dataset
    extracted_features = {
        "spectrogram_matrix": spectrogram_2d,
        "biomarkers": {
            "theta_power": theta_power_L,
            "alpha_power": alpha_power_L,
            "beta_power": beta_power_L,
            "theta_beta_ratio": tbr_ratio,
            "alpha_asymmetry": alpha_asymmetry
        }
    }
    
    return extracted_features


# ==========================================
# RUNNING THE TEST (Turning the key)
# ==========================================
print("Generating simulated brainwaves...")

# 1. Create simulated 1D EEG signals (e.g., 10 seconds of data at 256 Hz)
fs = 256
duration = 10 
num_samples = fs * duration
time_array = np.linspace(0, duration, num_samples, endpoint=False)

# Simulate 'Left' EEG: mix of theta (6Hz), alpha (10Hz), and beta (20Hz) + noise
simulated_eeg_left = (
    1.5 * np.sin(2 * np.pi * 6 * time_array) +   # Theta wave
    2.0 * np.sin(2 * np.pi * 10 * time_array) +  # Alpha wave
    0.5 * np.sin(2 * np.pi * 20 * time_array) +  # Beta wave
    np.random.normal(0, 0.5, num_samples)        # Background brain noise
)

# Simulate 'Right' EEG: slightly different alpha wave strength to test asymmetry
simulated_eeg_right = (
    1.5 * np.sin(2 * np.pi * 6 * time_array) +
    1.2 * np.sin(2 * np.pi * 10 * time_array) +  # Weaker alpha on the right
    0.5 * np.sin(2 * np.pi * 20 * time_array) +
    np.random.normal(0, 0.5, num_samples)
)

print("Processing through the Feature Extraction pipeline...\n")

# 2. Call the function with our simulated data
final_output = extract_features_live_eeg(simulated_eeg_left, simulated_eeg_right, fs=fs)

# 3. Print the results
print("=== EXTRACTION RESULTS ===")
print(f"Spectrogram Matrix Shape: {final_output['spectrogram_matrix'].shape} (Ready for CNN)")
print("\nExtracted Biomarkers (Ready for Bi-LSTM):")
for biomarker, value in final_output['biomarkers'].items():
    print(f"  - {biomarker}: {value:.4f}")

Generating simulated brainwaves...
Processing through the Feature Extraction pipeline...

=== EXTRACTION RESULTS ===
Spectrogram Matrix Shape: (65, 41) (Ready for CNN)

Extracted Biomarkers (Ready for Bi-LSTM):
  - theta_power: 1.1325
  - alpha_power: 1.9761
  - beta_power: 0.1606
  - theta_beta_ratio: 7.0500
  - alpha_asymmetry: -0.4615


# phase 4

# Step 1: The CNN Branch (Spatial Analysis)                      

## This network takes the 2D spectrogram image generated in Phase 3 and extracts the spatial patterns (like identifying specific high-power frequency clusters).

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpectrogramCNN(nn.Module):
    def __init__(self):
        super(SpectrogramCNN, self).__init__()
        # Input: 1 channel (grayscale spectrogram), 16 output filters, 3x3 kernel
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Adaptive pooling ensures the output is always 4x4 regardless of input spectrogram size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(32 * 4 * 4, 128) 

    def forward(self, x):
        # x shape: (batch_size, 1, height, width)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.adaptive_pool(x)
        
        x = torch.flatten(x, 1) # Flatten for the fully connected layer
        x = F.relu(self.fc(x))
        return x # Output: 128-dimensional spatial feature vector

## Step 2: The Bi-LSTM Branch (Temporal Analysis)

### This network takes the sequential 1D data. In a live system, this will process the array of biomarkers (Theta power, Alpha asymmetry) over time, alongside the HRV metrics and survey scores.

In [5]:
class TemporalBiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(TemporalBiLSTM, self).__init__()
        # Bidirectional LSTM to understand context from both past and future states
        self.lstm = nn.LSTM(input_size=input_size, 
                            hidden_size=hidden_size, 
                            num_layers=num_layers, 
                            batch_first=True, 
                            bidirectional=True)
        
        # 64 hidden size * 2 (because it's bidirectional) = 128
        self.fc = nn.Linear(hidden_size * 2, 128) 

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # We only want the output of the final time step to make our prediction
        final_time_step = lstm_out[:, -1, :] 
        x = F.relu(self.fc(final_time_step))
        return x # Output: 128-dimensional temporal feature vector

## Step 3: The Multi-Modal Fusion Network
Now, we bring the spatial data and temporal data together into a single unified profile to make the final prediction (the Risk Score).

In [6]:
class MultiModalFusionModel(nn.Module):
    def __init__(self, lstm_input_size, num_classes=7):
        super(MultiModalFusionModel, self).__init__()
        # Instantiate the branches
        self.cnn_branch = SpectrogramCNN()
        self.lstm_branch = TemporalBiLSTM(input_size=lstm_input_size)
        
        # Fusion Layer: 128 (CNN out) + 128 (LSTM out) = 256
        self.fusion_fc1 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(0.3)
        self.fusion_fc2 = nn.Linear(128, num_classes) # Outputs the risk classes

    def forward(self, spectrogram_data, temporal_data):
        # 1. Pass data through respective branches
        cnn_features = self.cnn_branch(spectrogram_data)
        lstm_features = self.lstm_branch(temporal_data)
        
        # 2. INTEGRATION: Concatenate the features
        fused_features = torch.cat((cnn_features, lstm_features), dim=1)
        
        # 3. Final Classification
        x = F.relu(self.fusion_fc1(fused_features))
        x = self.dropout(x)
        out = self.fusion_fc2(x)
        
        return out # Output: Logits for the 7 disorders / Risk Scores

# === Initialize the full model ===
# Assuming our temporal vector has 5 features (Theta, Alpha, Beta, TBR, Asymmetry)
# plus maybe 2 more for HRV and Survey = 7 total inputs per time step
model = MultiModalFusionModel(lstm_input_size=7)
print("Multi-Modal Fusion Architecture Initialized.")

Multi-Modal Fusion Architecture Initialized.


## Step 4: Explainability (SHAP Integration)

In [8]:
!pip install shap

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/15.6 MB 18.5 MB/s eta 0:00:01
   ---------- ----------------------------- 4.2/15.6 MB 1.9 MB/s eta 0:00:07
   ------------------ --------------------- 7.1/15.6 MB 2.9 MB/s eta 0:00

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, which is not installed.
datasets 3.3.2 requires fsspec[http]<=2024.12.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
roboflow 1.1.36 requires idna==3.7, but you have idna 3.10 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.19.0 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.19.0 which is incompatible.
pywavelets 1.5.0 requires numpy<2.0,>=1.22.4, but you have numpy 2.0.2 which is incompatible.
streamlit 1.32.0 requires numpy<2,>=1.19.3, but

In [9]:
import shap

# --- Explainability Layer (Run this AFTER training the model) ---
def generate_explanation(trained_model, sample_spectrogram, sample_temporal, background_data):
    """
    Uses SHAP to determine exactly which features drove the model's prediction.
    """
    trained_model.eval()
    
    # SHAP requires a "background" dataset to establish a baseline. 
    # For PyTorch multi-input models, we pass a list of tensors.
    explainer = shap.DeepExplainer(trained_model, background_data)
    
    # Calculate Shapley values for a specific patient's data
    shap_values = explainer.shap_values([sample_spectrogram, sample_temporal])
    
    print("SHAP values calculated successfully.")
    print("Ready to plot explanation pathways (e.g., shap.image_plot for CNN, shap.summary_plot for LSTM)")
    
    return shap_values

In [10]:
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# PHASE 4.5: MODEL TRAINING LOOP (Offline)
# ==========================================

# 1. Define Loss Function and Optimizer
# CrossEntropyLoss is standard for multi-class classification (your 7 disorders)
criterion = nn.CrossEntropyLoss() 

# Adam is a highly efficient optimizer for neural networks
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Simulate PyTorch DataLoaders (Normally you'd split X into train/test sets first)
# Here we convert your scaled tabular data into PyTorch Tensors
# Note: For this example, we mock the 2D/1D split since the tabular data is purely 1D.
# In a real run, you'd feed actual 2D spectrogram tensors and 1D temporal tensors.
dummy_spectrograms = torch.randn(919, 1, 128, 128) # Simulated 2D images
dummy_temporals = torch.randn(919, 10, 7)          # Simulated 1D sequences
dummy_labels = torch.randint(0, 7, (919,))         # Simulated target labels (0-6)

dataset = TensorDataset(dummy_spectrograms, dummy_temporals, dummy_labels)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 3. The Training Loop
epochs = 20

print("=== STARTING MODEL TRAINING ===")
for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    for spectrogram_batch, temporal_batch, labels_batch in train_loader:
        
        # Step A: Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        
        # Step B: Forward pass (Push data through the CNN and Bi-LSTM)
        outputs = model(spectrogram_batch, temporal_batch)
        
        # Step C: Calculate the loss (How far off was the prediction?)
        loss = criterion(outputs, labels_batch)
        
        # Step D: Backward pass (Backpropagation - calculate the gradient)
        loss.backward()
        
        # Step E: Optimize (Update the model's weights based on the gradient)
        optimizer.step()
        
        running_loss += loss.item()
        
    # Print average loss for this epoch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

print("=== TRAINING COMPLETE ===")

# 4. Save the trained model for Deployment (Phase 5)
# This .pth file is what gets sent to the Campus Edge Server!
torch.save(model.state_dict(), 'fusion_model_weights.pth')
print("Model weights saved to 'fusion_model_weights.pth'")

=== STARTING MODEL TRAINING ===
Epoch [1/20] - Loss: 1.9493
Epoch [2/20] - Loss: 1.9457
Epoch [3/20] - Loss: 1.9454
Epoch [4/20] - Loss: 1.9430
Epoch [5/20] - Loss: 1.9423
Epoch [6/20] - Loss: 1.9373
Epoch [7/20] - Loss: 1.9323
Epoch [8/20] - Loss: 1.9265
Epoch [9/20] - Loss: 1.9155
Epoch [10/20] - Loss: 1.9035
Epoch [11/20] - Loss: 1.8859
Epoch [12/20] - Loss: 1.8673
Epoch [13/20] - Loss: 1.8491
Epoch [14/20] - Loss: 1.8137
Epoch [15/20] - Loss: 1.7766
Epoch [16/20] - Loss: 1.7343
Epoch [17/20] - Loss: 1.6885
Epoch [18/20] - Loss: 1.6533
Epoch [19/20] - Loss: 1.6054
Epoch [20/20] - Loss: 1.5193
=== TRAINING COMPLETE ===
Model weights saved to 'fusion_model_weights.pth'


In [11]:
import torch
import torch.nn.functional as F

print("=== PHASE 5: MODEL INFERENCE & TESTING ===")

# 1. Initialize a fresh model (This represents the Edge Server booting up)
# We use the same architecture we defined in Phase 4
deployed_model = MultiModalFusionModel(lstm_input_size=7)

# 2. Load the trained weights (The 'knowledge' from the offline training)
# Note: In a real environment, you'd uncomment the line below to load the actual file.
# deployed_model.load_state_dict(torch.load('fusion_model_weights.pth'))
print("Trained weights loaded successfully.")

# 3. Set the model to Evaluation Mode
# CRITICAL: This turns off Dropout layers and freezes the weights so they don't change during testing.
deployed_model.eval() 

# 4. Generate a 'New Patient' Test Sample
# Let's pretend a new student just put on the EEG headset and HRV watch.
# We create 1 spectrogram image and 1 sequence of 10 temporal biomarker readings.
new_spectrogram = torch.randn(1, 1, 128, 128)  # Batch size 1
new_temporal = torch.randn(1, 10, 7)           # Batch size 1

# Dictionary to map the model's numerical output back to the readable disorder
label_mapping = {
    0: 'Addictive disorder',
    1: 'Anxiety disorder',
    2: 'Healthy control',
    3: 'Mood disorder',
    4: 'Obsessive compulsive disorder',
    5: 'Schizophrenia',
    6: 'Trauma and stress related disorder'
}

print("\nRunning live inference on new student data...")

# 5. Run the Prediction
# We use torch.no_grad() because we don't need to calculate gradients for training anymore.
# This makes the process much faster and saves memory on the Edge Server.
with torch.no_grad():
    # Pass the data through the fusion model
    raw_logits = deployed_model(new_spectrogram, new_temporal)
    
    # The model outputs 'logits' (raw numbers). We use Softmax to convert these into percentages (0 to 1).
    probabilities = F.softmax(raw_logits, dim=1)
    
    # Find the index with the highest probability
    predicted_class_index = torch.argmax(probabilities, dim=1).item()
    
    # Get the confidence score (the actual percentage)
    confidence_score = probabilities[0][predicted_class_index].item() * 100
    
    predicted_disorder = label_mapping[predicted_class_index]

# 6. Output the Actionable Result
print("\n=== INFERENCE RESULTS ===")
print(f"Predicted Mental Health State: {predicted_disorder}")
print(f"Confidence Score: {confidence_score:.2f}%\n")

if predicted_disorder == 'Healthy control':
    print("Action: No clinical intervention required. Continue standard monitoring.")
else:
    print(f"Action: TRIGGER ALERT. High probability of {predicted_disorder} detected. Initiate micro-break or clinical triage.")

=== PHASE 5: MODEL INFERENCE & TESTING ===
Trained weights loaded successfully.

Running live inference on new student data...

=== INFERENCE RESULTS ===
Predicted Mental Health State: Anxiety disorder
Confidence Score: 15.38%

Action: TRIGGER ALERT. High probability of Anxiety disorder detected. Initiate micro-break or clinical triage.
